# **Recolección de Datos**

## Dataset: MIT-BIH Arrhythmia Database

Para este proyecto usamos el dataset **MIT-BIH Arrhythmia Database**, disponible
públicamente en PhysioNet (https://physionet.org/content/mitdb/1.0.0/).

Lo elegimos por varias razones:
- Es el dataset de referencia en investigación de arritmias cardíacas, usado en
  cientos de estudios publicados.
- Los latidos están anotados manualmente por cardiólogos, lo que garantiza la
  calidad de las etiquetas.
- Es público, gratuito y anonimizado, por lo que no hay restricciones éticas
  ni legales para su uso.

## Características técnicas

- **48 registros** de pacientes diferentes
- **Frecuencia de muestreo:** 360 Hz (360 muestras por segundo)
- **Duración por registro:** ~30 minutos
- **Derivaciones:** 2 por registro (usamos la primera, MLII)
- **Total de latidos anotados:** ~110.000
- **Tipos de anotaciones:** 18 clases diferentes (latidos normales,
  ventriculares, supraventriculares, bloqueos, etc.)

## Descarga de los datos

Usamos la librería `wfdb` para descargar los registros directamente desde
PhysioNet sin necesidad de descargar manualmente los archivos.

Descargamos 36 de los 48 registros disponibles, seleccionando los que tienen
mayor variedad de tipos de arritmias.

In [2]:
!pip install wfdb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 77.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.


In [3]:
import wfdb
import os

# Lista de registros que descargamos
records = [
    "100","101","102","103","104","105","106","107","108","109",
    "111","112","113","114","115","116","117","118","119",
    "121","122","123","124","200","201","202","203","205","207",
    "208","209","210","212","213","214"
]

# Descarga desde PhysioNet
wfdb.dl_database("mitdb", dl_dir="mitdb", records=records)

# Verificamos que se han descargado correctamente
print(os.listdir("mitdb"))

Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record list for: 111
Generating record list for: 112
Generating record list for: 113
Generating record list for: 114
Generating record list for: 115
Generating record list for: 116
Generating record list for: 117
Generating record list for: 118
Generating record list for: 119
Generating record list for: 121
Generating record list for: 122
Generating record list for: 123
Generating record list for: 124
Generating record list for: 200
Generating record list for: 201
Generating record list for: 202
Generating record list for: 203
Generating record list for: 205
Generating record list for: 207
Generating record list for: 208
Generating record list for: 209
Generati

## Estructura de los archivos descargados

Cada registro se compone de tres archivos:

- `.dat` — señal eléctrica en formato binario
- `.hea` — metadatos del registro (frecuencia, número de canales, etc.)
- `.atr` — anotaciones de los cardiólogos (posición y tipo de cada latido)

## Verificación de los datos


Antes de pasar al preprocesamiento, comprobamos que los registros se leen
correctamente y revisamos la información básica de cada uno.

In [4]:
# Leemos un registro de ejemplo para verificar la estructura
record     = wfdb.rdrecord("mitdb/100")
annotation = wfdb.rdann("mitdb/100", "atr")

print("Frecuencia de muestreo:", record.fs, "Hz")
print("Duración (seg):", len(record.p_signal) / record.fs)
print("Forma de la señal:", record.p_signal.shape)
print("Número de anotaciones:", len(annotation.sample))
print("Tipos de latidos en este registro:", set(annotation.symbol))

Frecuencia de muestreo: 360 Hz
Duración (seg): 1805.5555555555557
Forma de la señal: (650000, 2)
Número de anotaciones: 2274
Tipos de latidos en este registro: {'A', '+', 'N', 'V'}


## Consideraciones sobre los datos

- El dataset está **anonimizado**: no contiene nombres, fechas ni ningún dato
  identificativo de los pacientes.
- Las señales son datos reales recogidos en el Beth Israel Hospital de Boston
  entre 1975 y 1979.
- Algunos registros incluyen arritmias poco frecuentes grabadas intencionalmente
  para que el dataset sea representativo de casos clínicos difíciles.